In [1]:
import pandas as pd
import numpy as np

In [7]:
# function to set working directory to the git repository root
def set_working_directory_to_git_root():
    import os
    import subprocess

    # Get the current working directory
    current_dir = os.getcwd()

    # Use git command to find the root directory of the repository
    try:
        git_root = subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], cwd=current_dir).strip().decode('utf-8')
        os.chdir(git_root)
        print(f"Working directory set to git repository root: {git_root}")
    except subprocess.CalledProcessError as e:
        print("Error: Not a git repository or git is not installed.")
        print(e)

set_working_directory_to_git_root()

Working directory set to git repository root: /mnt/tigrlab/projects/aabdulrasul/TAY/PhD_1


In [6]:
## demographics

TAY_core = pd.read_csv('CLIN/data/datacut_demographics_core_data.csv')[['subject_id',  'age_enrolment',
       'age_first_assessment_in_month', 'assigned_sex_at_birth', 'prodromal_psychosis', 'gender_diversity', 'gender_identity',
       'ethnicity',
       'family_income' ]]

## from youth report get data on parents education and income


## from caregiver report get data on parents education and income
    # demo_ed_q1 = employment status (primary)
    # demo_ed_q2 = employment status (secondary)
    # demo_ed_q2_nl = employment status (secondary) free text
    # demo_ed_q3 = Are you currently a student?
    # demo_ed_q3_nl = Are you currently a student? free text
    # demo_ed_q4 = What is the highest level of education you have completed? 
    # demo_ed_q4_nl = What is the highest level of education you have completed? free text

TAY_caregiver = pd.read_csv('CLIN/data/demographic_caregiver_data.csv')[['subject_id', 'demo_ed_q1', 'demo_ed_q2', "demo_ed_q2_nl", 'demo_ed_q3', 'demo_ed_q4', 'demo_ed_q4_nl']]

## dictionary to convert demo_ed_q4 answers to a common format 
education_dict = {1: 'Some elementary school', 2: 'Some high school', 3: 'High school diploma', 4: 'Trade/vocational/technical certificate/ apprenticeship', 5: 'Some college/technical institute', 6: 'Some University', 7: 'Graduated College', 8: 'Graduated University', 9: 'Certificate college/university', 10: 'Masters degree', 11: 'Doctorate degree', 12: 'Prefer not to answer', 13: 'Not listed (please specify)'}

# convert demo_ed_q4 to a common format using the dictionary
TAY_caregiver['parental_education_level'] = TAY_caregiver['demo_ed_q4'].map(education_dict)

# left join TAY_core and TAY_caregiver on subject_id
TAY_core = pd.merge(TAY_core, TAY_caregiver[['subject_id', 'parental_education_level']], on='subject_id', how='left')

In [3]:
ASEBA_youth = pd.read_csv('/home/hassan/Documents/Kimel/PhD_1/CLIN/data/aseba/aseba_youth_selfreport_ysr_data.csv')[['subject_id',  "ysr_fworktype", "ysr_mworktype"]]

TAY_parent_work = pd.merge(TAY_caregiver, ASEBA_youth[['subject_id',  "ysr_fworktype", "ysr_mworktype"]], on='subject_id', how='left')

# if there are any values in ysr_fworktype or ysr_mworktype or demo_ed_q1 or demo_ed_q1, then make caregiver_employment = 'Employed'
#  unless ysr_fworktype or ysr_mworktype is 'N.A.', 'MI', 'ASKU', 'UNK', PNA', 'NI then make it NA however, if there are values in demo_ed_q1 or demo_ed_q2 then make caregiver_education_level = 'Employed'  if the values are 2
# 1. Define the list of invalid/missing responses
invalid_work = ['N.A.', 'MI', 'ASKU', 'UNK', 'PNA', 'NI', "Not working", "NASK"]

TAY_parent_work['demo_ed_q1'] = pd.to_numeric(TAY_parent_work['demo_ed_q1'], errors='coerce')
TAY_parent_work['demo_ed_q2'] = pd.to_numeric(TAY_parent_work['demo_ed_q2'], errors='coerce')

# Create boolean conditions for each of your rules
# Check if youth report has valid father/mother work types
valid_fwork = TAY_parent_work['ysr_fworktype'].notna() & ~TAY_parent_work['ysr_fworktype'].isin(invalid_work)
valid_mwork = TAY_parent_work['ysr_mworktype'].notna() & ~TAY_parent_work['ysr_mworktype'].isin(invalid_work)

# Check if caregiver report indicates employment (value == 2)
valid_ed_q1 = TAY_parent_work['demo_ed_q1'] == 2
valid_ed_q2 = TAY_parent_work['demo_ed_q2'] == 2

#  Combine all conditions with OR (|)
is_employed = valid_fwork | valid_mwork | valid_ed_q1 | valid_ed_q2

#Assign the results to the new column
# If is_employed is True -> 'Employed', Else -> pd.NA
TAY_parent_work['caregiver_employment'] = np.where(is_employed, 'Employed', pd.NA)


# 2. Define conditions for Unemployment
# We check if either q1 or q2 is 4 AND they aren't already flagged as employed 
# (e.g., if one parent is 2 and the other is 4, they stay 'Employed')
is_unemployed = (TAY_parent_work['demo_ed_q1'] == 4) | (TAY_parent_work['demo_ed_q2'] == 4)

#  Use np.select to apply the hierarchy
conditions = [
    is_employed,        # Condition 1
    is_unemployed       # Condition 2
]

choices = [
    'Employed',         # Choice for Condition 1
    'Unemployed'        # Choice for Condition 2
]

# If neither condition is met, it defaults to pd.NA
TAY_parent_work['caregiver_employment'] = np.select(conditions, choices, default=pd.NA)

In [4]:
## make TAY_demo

TAY_demo = pd.merge(TAY_core, TAY_parent_work[['subject_id', 'caregiver_employment']], on='subject_id', how='left')